# Running the Granite TSFM ensemble on GIFT-Eval

**This notebook is intended to reproduce the submitted GIFT-Eval ensemble results.**

It installs GIFT-Eval and the permanent Granite TSFM ensemble branch, loads the experiment configuration stored with the Granite benchmark runner, runs the benchmark, and validates the generated `all_results.csv`.

## Installation

Run this notebook from within a GIFT-Eval checkout. The Granite checkout is reused if it already exists. Restart the kernel after changing an installed checkout or package version.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

gift_eval_root = Path.cwd().resolve().parent
notebook_dir = Path.cwd().resolve()
if not (gift_eval_root / "pyproject.toml").is_file() or not (gift_eval_root / "src/gift_eval").is_dir():
    raise RuntimeError("Run this notebook from the GIFT-Eval notebooks directory")

granite_source = notebook_dir / "granite-tsfm"
if not os.path.exists("granite-tsfm"):
    !git clone --branch granite_tsfm_ensemble --single-branch --depth 1 https://github.com/ibm-granite/granite-tsfm.git
else:
    print("Folder 'granite-tsfm' already exists. Skipping git clone.")

In [ ]:
%cd granite-tsfm
%pip install -q -e ".[notebooks]"
%cd ..
%pip install -q -e "..[baseline]"

sys.path.insert(0, str(granite_source.resolve()))
sys.path.insert(0, str((granite_source / "notebooks/hfdemo/gift_eval_ensemble").resolve()))

## Data and experiment configuration

Set `GIFT_EVAL_PATH` directly below or set the `GIFT_EVAL` environment variable. The path must point to the downloaded GIFT-Eval data.

In [ ]:
import json
from dotenv import load_dotenv

load_dotenv(gift_eval_root / ".env")

# Set this to an explicit path, or leave it as None to use GIFT_EVAL.
GIFT_EVAL_PATH = None
if GIFT_EVAL_PATH is None:
    GIFT_EVAL_PATH = os.environ.get("GIFT_EVAL")
if not GIFT_EVAL_PATH or not Path(GIFT_EVAL_PATH).expanduser().is_dir():
    raise RuntimeError(
        "Set GIFT_EVAL_PATH or GIFT_EVAL to the downloaded dataset directory"
    )
os.environ["GIFT_EVAL"] = str(Path(GIFT_EVAL_PATH).expanduser().resolve())

config_path = granite_source / "notebooks/hfdemo/gift_eval_ensemble/experiment_config.json"
if not config_path.is_file():
    raise FileNotFoundError(f"Missing experiment configuration: {config_path}")
experiment_config = json.loads(config_path.read_text())
from run_gift_eval import run_evaluation

In [ ]:
SETTINGS = experiment_config["defaults"].copy()
SETTINGS["model_name_config"] = "iqr-weighted-ensemble-ibm-tsfm-granite-pt"
MODEL_NAME = SETTINGS["model_name_config"]
SETTINGS.update(
    skip_processed=True,
    save_member_results=True,
)
output_root = Path(os.environ.get("GIFT_EVAL_OUTPUT_ROOT", gift_eval_root / "outputs/granite_tsfm_iqr_ensemble")).expanduser().resolve()

print(f"Model: {MODEL_NAME}")
print(f"Data: {os.environ['GIFT_EVAL']}")
print(json.dumps(experiment_config["configurations"][MODEL_NAME], indent=2))

## Environment record

In [ ]:
import platform

import numpy as np
import pandas as pd
import torch
import transformers

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available")
print("Granite revision:", subprocess.run(["git", "-C", str(granite_source), "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip())
print("GIFT-Eval revision:", subprocess.run(["git", "-C", str(gift_eval_root), "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip())

## Smoke test

This verifies data access, model loading, ensemble inference, and metric generation without modifying the submitted result.

In [ ]:
smoke_settings = {
    **SETTINGS,
    "datasets": ["us_births/M"],
    "out_dir": output_root / "smoke",
}
smoke_csv = run_evaluation(**smoke_settings)
display(pd.read_csv(smoke_csv).sort_values("dataset"))

## Full benchmark

Set `RUN_FULL_BENCHMARK=True` after the smoke test succeeds, or set the environment variable `RUN_FULL_BENCHMARK=true` for a batch run. The run is resumable and writes beneath the ignored `outputs/` directory, leaving the committed leaderboard result unchanged.

In [ ]:
RUN_FULL_BENCHMARK = os.environ.get("RUN_FULL_BENCHMARK", "false").lower() in {"1", "true", "yes"}
full_csv = output_root / "full" / MODEL_NAME / SETTINGS["out_name"]
if RUN_FULL_BENCHMARK:
    full_settings = {
        **SETTINGS,
        "datasets": None,
        "out_dir": output_root / "full",
    }
    full_csv = run_evaluation(**full_settings)
else:
    print("Full evaluation not started.")

## Validate and compare results

The member-result checks ensure that no configured model was silently omitted. When the submitted CSV exists, it is compared with the regenerated result using a small numerical tolerance.

In [ ]:
if full_csv.is_file():
    results = pd.read_csv(full_csv).sort_values("dataset").reset_index(drop=True)
    expected_rows = (
        len(experiment_config["datasets"]["short"])
        + 2 * len(experiment_config["datasets"]["medium_long"])
    )
    metrics = results.filter(regex=r"^eval_metrics/")
    assert len(results) == results["dataset"].nunique() == expected_rows
    assert results.shape[1] == 15
    assert results["model"].eq(MODEL_NAME).all()
    assert metrics.shape[1] == 11 and np.isfinite(metrics.to_numpy()).all()

    expected_members = experiment_config["configurations"][MODEL_NAME]["model_names"]
    for member_name in expected_members:
        member_csv = full_csv.parent / "members" / member_name / SETTINGS["out_name"]
        assert member_csv.is_file(), f"Missing member results: {member_name}"
        member_results = pd.read_csv(member_csv)
        assert len(member_results) == member_results["dataset"].nunique() == expected_rows

    submitted_csv = gift_eval_root / "results" / MODEL_NAME / "all_results.csv"
    if submitted_csv.is_file():
        submitted = pd.read_csv(submitted_csv).sort_values("dataset").reset_index(drop=True)
        pd.testing.assert_frame_equal(
            results, submitted, check_dtype=False, rtol=1e-5, atol=1e-7
        )
        print(f"Generated metrics match {submitted_csv}")
    else:
        print(f"Copy the validated CSV to {submitted_csv} for submission.")
    display(results)
else:
    print("Run the full benchmark before validating results.")